# Phase 2: Synthetic Credit-Limit Uplift Walkthrough

This notebook runs the Phase 2 modeling pipeline on the synthetic RCT:

1. Prep `synthetic_credit_full.parquet` (no ground truth in training)
2. Fit **T-learner** and **CausalForestDML**
3. Evaluate with **Qini / uplift-at-k** (policy metrics) and **PEHE / per-segment CATE recovery** (oracle metrics only possible because we simulated the DGP)

Phase 1 (Hillstrom) stays untouched under the repo root `/src` — this phase lives entirely under `phase2_synthetic_credit/`.

## Why accuracy / AUC are still the wrong metrics

Same logic as Phase 1, now with a credit framing:

| Framing | Question | Metric |
|---|---|---|
| Classification | "Who stays in good standing?" | Accuracy, AUC |
| Uplift / CATE | "Whose good-standing probability **changes because of** the limit increase?" | Qini, uplift-at-k, PEHE |

**Sure Things** stay good with or without the increase — a classifier loves them; uplift is ~0.  
**Lost Causes** fail either way — classifier ranks them low; uplift is still ~0.  
**Persuadables** improve *because* of the increase — that is who we want to treat.  
**Sleeping Dogs** may get *worse* if treated (negative CATE) — ranking by outcome risk alone will not flag them.

Phase 2 adds something Hillstrom could not: **known `true_cate`**, so we can measure whether the model recovers the answer key (PEHE), not only whether its ranking beats random (Qini).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

## 1. Data prep

Treatment was generated **independently** of segment and covariates (`Bernoulli(0.5)`).  
Ground-truth columns never enter this step.

In [ ]:
from data_prep import run_data_prep, feature_columns

train_df, test_df = run_data_prep()
print("n features:", len(feature_columns(train_df)))
train_df.head()

## 2. T-learner

μ₁(x) on treated, μ₀(x) on control; ĈATE(x) = μ₁(x) − μ₀(x).  
Default base learner: `GradientBoostingClassifier`.

In [ ]:
from models.t_learner import run_t_learner

t_preds = run_t_learner(model_name="gbm")
t_preds.head()

## 3. CausalForestDML

Chosen over LinearDML for heterogeneous segment CATEs. Import/fit failures raise — no silent fallback.

In [ ]:
from models.causal_forest import run_causal_forest

cf_preds = run_causal_forest()
cf_preds.head()

## 4. Policy metrics (Qini / uplift-at-k)

These use only observed outcomes + treatment on the test set — the same metrics you would compute in a real RCT without an answer key.

In [ ]:
from evaluation.uplift_metrics import run_evaluation

uplift_summary = run_evaluation()
uplift_summary

## 5. CATE recovery (PEHE + per-segment)

Join predictions to `synthetic_credit_ground_truth.parquet` **only here**.

- **PEHE** = mean((predicted_cate − true_cate)²) — lower is better
- **Per-segment means** — did we recover Persuadables (positive) vs Sleeping Dogs (negative)?

In [ ]:
from evaluation.cate_recovery import run_cate_recovery

overall_df, segment_df = run_cate_recovery()
segment_df

In [ ]:
# Plain-English summary: did we recover the latent segments?
def _seg_means(model: str) -> dict[str, float]:
    sub = segment_df[segment_df["model"] == model]
    return {
        r["segment"]: float(r["mean_predicted_cate"])
        for _, r in sub.iterrows()
    }

t_pehe = float(overall_df.loc[overall_df["model"] == "T-learner", "pehe"].iloc[0])
cf_pehe = float(overall_df.loc[overall_df["model"] == "CausalForestDML", "pehe"].iloc[0])
better = "T-learner" if t_pehe <= cf_pehe else "CausalForestDML"

t_means = _seg_means("T-learner")
cf_means = _seg_means("CausalForestDML")

print("=== Plain-English summary ===\n")
print(
    f"On PEHE (CATE recovery), {better} wins "
    f"(T-learner PEHE={t_pehe:.4f}, CausalForestDML PEHE={cf_pehe:.4f}). "
    f"Qini/uplift numbers are in the table above for the policy view."
)
print()
print(
    f"Segment recovery (mean predicted CATE): "
    f"T-learner Persuadables={t_means.get('Persuadables', float('nan')):+.3f}, "
    f"Sure Things={t_means.get('Sure Things', float('nan')):+.3f}, "
    f"Lost Causes={t_means.get('Lost Causes', float('nan')):+.3f}, "
    f"Sleeping Dogs={t_means.get('Sleeping Dogs', float('nan')):+.3f}. "
    f"CausalForestDML Persuadables={cf_means.get('Persuadables', float('nan')):+.3f}, "
    f"Sure Things={cf_means.get('Sure Things', float('nan')):+.3f}, "
    f"Lost Causes={cf_means.get('Lost Causes', float('nan')):+.3f}, "
    f"Sleeping Dogs={cf_means.get('Sleeping Dogs', float('nan')):+.3f}."
)
print()
print(
    "A successful run should show clearly positive mean CATE on Persuadables, "
    "near-zero on Sure Things and Lost Causes, and clearly negative on Sleeping Dogs - "
    "i.e., the models recover the DGP's answer key, not just overall good_standing risk. "
    "That is the Phase 2 proof point before trusting the same estimators on real credit data "
    "where true_cate is unknowable."
)

## 6. Closing check: who does top-decile targeting actually reach?

Per-segment *mean* CATE can look right while a top-decile mailing list is still contaminated. This final check asks the operational question: **if we treat the top 10% by predicted CATE, what true latent segments do we hit?** Ground truth is joined here only for reporting — never for training.

This is the last planned Phase 2 analysis before write-up.

In [ ]:
from evaluation.segment_recovery_check import run_segment_recovery_check

seg_recovery, seg_ops_summaries = run_segment_recovery_check()
# Enrichment table (top/bottom x true segment), excluding meta rows
enrichment_table = seg_recovery[
    seg_recovery["true_segment"].isin(
        ["Persuadables", "Sure Things", "Lost Causes", "Sleeping Dogs"]
    )
].copy()
enrichment_table

### Phase 2 closing summary (operational targeting)

Targeting by predicted CATE **does** find true Persuadables and avoid true Sleeping Dogs on this simulated RCT. Against a ~20% Persuadables / ~25% Sleeping Dogs population mix, the **T-learner** top decile is **95.5% Persuadables** (~4.9× enrichment) with only **0.1% Sleeping Dogs**; its bottom decile is **94.8% Sleeping Dogs**. **CausalForestDML** is even cleaner: top decile is **100% Persuadables** (0% Sleeping Dogs) and bottom decile is **100% Sleeping Dogs**.

On the costly-mistake metric — **Sleeping Dogs contamination in the top decile** — **CausalForestDML is operationally safer** (0.00% vs T-learner 0.12%). That aligns with its PEHE win, not with T-learner’s slight Qini edge: for a real “who gets the credit increase” decision, prefer the model that keeps harmful reactors out of the treated list, not only the one with the prettier ranking curve.

Phase 2 is complete: the DGP’s answer key is recoverable, and top-decile deployment maps cleanly onto Persuadables while suppressing Sleeping Dogs. Next step is project write-up / README, not more Hillstrom or synthetic analysis.